# SimWorld Studio

**AI-powered 3D scene generation platform** — chat with Claude to build urban scenes in real-time.

## What this notebook does
1. Checks your Colab GPU
2. Downloads the minimal SimWorld binary (~15 GB)
3. Installs SimWorld Studio platform
4. Sets up Claude authentication (API key or Claude Code login)
5. Launches everything and gives you a browser URL

**Run all cells in order.** Total setup time: ~5 minutes.

---

## Cell 1: GPU Check

In [1]:
"""Verify GPU is available — T4 or better required."""
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU (T4)")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

supported = ["T4", "A100", "V100", "L4", "A10"]
if not any(g in gpu_info for g in supported):
    print(f"WARNING: Unrecognized GPU. Supported: {supported}")
    print("Continuing anyway...")
else:
    print("GPU check passed!")

GPU: Tesla T4, 15360 MiB
GPU check passed!


In [2]:
import os
os.environ["PATH"] = "/tools/node/bin:" + os.environ["PATH"]

## Cell 2: Download SimWorld (Minimal Binary)

Downloads a minimal UE Editor binary (~15 GB compressed) with just the assets needed for the Studio demo.

In [3]:
"""Download and extract the minimal SimWorld binary."""
import os

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
SIMWORLD_ARCHIVE = "/tmp/SimWorld-Studio-Minimal.tar.gz"
DOWNLOAD_URL = "https://huggingface.co/datasets/SimWorld-AI/SimWorld-Studio/resolve/main/SimWorld-Studio-Minimal.tar.gz"

UE_BINARY = os.path.join(SIMWORLD_DIR, "Engine", "Binaries", "Linux", "UnrealEditor")

if os.path.exists(UE_BINARY):
    print(f"SimWorld binary already installed at: {SIMWORLD_DIR}")
else:
    print("Downloading SimWorld minimal binary (~15 GB)...")
    !wget -q --show-progress -O {SIMWORLD_ARCHIVE} {DOWNLOAD_URL}
    print("Extracting...")
    !tar xzf {SIMWORLD_ARCHIVE} -C /content/
    !rm -f {SIMWORLD_ARCHIVE}

    if os.path.exists(UE_BINARY):
        os.chmod(UE_BINARY, 0o755)
        # Also chmod the launch script
        launch_sh = os.path.join(SIMWORLD_DIR, "SimWorld-Studio.sh")
        if os.path.exists(launch_sh):
            os.chmod(launch_sh, 0o755)
        print(f"SimWorld binary installed at: {SIMWORLD_DIR}")
    else:
        print("ERROR: UnrealEditor not found after extraction!")
        !find {SIMWORLD_DIR} -maxdepth 3 -type f -name "UnrealEditor*" | head -5

print(f"\nUE Binary: {UE_BINARY}")
print(f"Project: {os.path.join(SIMWORLD_DIR, 'gym_citynav', 'gym_citynav.uproject')}")

/tmp/SimWorld-Studi 100%[===================>]  14.13G   106MB/s    in 86s     
Extracting...
SimWorld binary installed at: /content/SimWorld-Studio-Minimal

UE Binary: /content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor
Project: /content/SimWorld-Studio-Minimal/gym_citynav/gym_citynav.uproject


## Cell 3: Install Studio Platform

In [4]:
"""Install the Studio platform + Claude Code CLI."""
import subprocess, shutil, os

# --- Studio platform (pip package from GitHub) ---
print("[1/3] Installing SimWorld Studio...")

STUDIO_GIT_URL = "git+https://github.com/SimWorld-AI/SimWorld-Studio.git#subdirectory=packaging"

result = subprocess.run(
    ["pip", "install", "-q", STUDIO_GIT_URL],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("  Installed from GitHub.")
else:
    print(f"  ERROR: Could not install package.")
    print(f"  {result.stderr[:200]}")
    print("  Please check the GitHub URL or install manually.")

# Verify installation
studio_installed = shutil.which("simworld-studio") is not None
if studio_installed:
    ver = subprocess.run(["simworld-studio", "version"], capture_output=True, text=True).stdout.strip()
    print(f"  {ver}")
else:
    print("  WARNING: simworld-studio CLI not found after install!")

# --- Node.js (usually pre-installed in Colab) ---
print("[2/3] Checking Node.js...")
if not shutil.which("node"):
    print("  Installing Node.js...")
    !apt-get install -y -qq nodejs npm
else:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Node.js {node_ver} found.")

# --- Claude Code CLI ---
print("[3/3] Installing Claude Code CLI...")
if not shutil.which("claude"):
    !npm install -g @anthropic-ai/claude-code 2>/dev/null | tail -2
    print("  Claude Code CLI installed.")
else:
    claude_ver = subprocess.run(["claude", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Claude Code CLI {claude_ver} already installed.")

print("\nAll dependencies installed!")

[1/3] Installing SimWorld Studio...
  Installed from GitHub.
  simworld-studio v0.2.0
[2/3] Checking Node.js...
  Node.js v20.19.0 found.
[3/3] Installing Claude Code CLI...
2 packages are looking for funding
  run `npm fund` for details
  Claude Code CLI installed.

All dependencies installed!


## Cell 4: Authenticate with Claude

You have **two options** to authenticate:

**Option A (recommended):** Run the cell below and enter your Anthropic API key. Get one at [console.anthropic.com](https://console.anthropic.com).

**Option B:** If you have a Claude account with Claude Code access, you can run `!claude login` in a new cell instead. This uses OAuth — no API key needed.

Your credentials stay **entirely local** in this Colab runtime and are never sent to SimWorld servers.

In [ ]:
# """Authenticate with Claude — API key or Claude Code login."""
# import subprocess, os

# # Check if already authenticated via claude login (OAuth)
# claude_auth = subprocess.run(["claude", "auth", "status"], capture_output=True, text=True)
# already_logged_in = claude_auth.returncode == 0 and "authenticated" in claude_auth.stdout.lower()

# if already_logged_in:
#     print("Already authenticated via Claude Code login!")
#     print("(To switch to API key, set ANTHROPIC_API_KEY below)")
# elif os.environ.get("ANTHROPIC_API_KEY"):
#     print(f"API key already set (starts with {os.environ['ANTHROPIC_API_KEY'][:10]}...)")
# else:
#     print("Choose authentication method:\n")
#     print("  Option A: Enter API key below")
#     print("  Option B: Run '!claude login' in a new cell\n")

#     from getpass import getpass
#     api_key = getpass("Enter your Anthropic API key (sk-ant-...), or press Enter to skip: ")

#     if api_key.strip():
#         os.environ["ANTHROPIC_API_KEY"] = api_key.strip()
#         if not api_key.startswith("sk-ant-"):
#             print("WARNING: Key doesn't start with 'sk-ant-'. Double-check your key.")
#             print("Get yours at: https://console.anthropic.com")
#         else:
#             print("API key set! (stored only in this runtime's memory)")
#     else:
#         print("No API key entered. Use '!claude login' in a new cell to authenticate via OAuth.")

Choose authentication method:

  Option A: Enter API key below
  Option B: Run '!claude login' in a new cell

No API key entered. Use '!claude login' in a new cell to authenticate via OAuth.


## Cell 5: Launch SimWorld + Studio

In [ ]:
"""Start SimWorld (headless GPU) and the Studio backend."""
import subprocess, time, os, socket, requests

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
UE_BINARY    = f"{SIMWORLD_DIR}/Engine/Binaries/Linux/UnrealEditor"
UE_REAL      = f"{SIMWORLD_DIR}/Engine/Binaries/Linux/UnrealEditor-real"
PROJECT_FILE = f"{SIMWORLD_DIR}/gym_citynav/gym_citynav.uproject"
MCP_PORT     = 55559

# --- [1/6] Install headless rendering dependencies ---
print("[1/6] Installing rendering dependencies...")
r = subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     "xvfb", "vulkan-tools", "mesa-vulkan-drivers",
     "libvulkan1", "libegl1", "libgles2", "libgl1"],
    capture_output=True
)
print("  Done." if r.returncode == 0 else f"  WARNING: exit {r.returncode}")

# --- [2/6] Start virtual display ---
print("[2/6] Starting virtual display...")
subprocess.run(["pkill", "-f", "Xvfb"], capture_output=True)
time.sleep(1)
subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x720x24"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
os.environ["DISPLAY"] = ":99"
time.sleep(2)
print("  Virtual display :99 started.")

# --- [3/6] Configure Vulkan + user + permissions ---
# Key facts learned the hard way:
#   - Mesa device_select layer hides NVIDIA GPU → "No Vulkan driver found!"
#   - UE5 refuses to run as root (SIGABRT)
#   - Studio calls `claude --dangerously-skip-permissions` which also fails as root
#   → Both UE and Studio must run as ue5user with correct Vulkan env
print("[3/6] Configuring Vulkan, user, and permissions...")

# NVIDIA Vulkan ICD with full library path
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
with open("/usr/share/vulkan/icd.d/nvidia_icd.json", "w") as f:
    f.write('{\n    "file_format_version" : "1.0.0",\n    "ICD": {\n        "library_path": "/usr/lib64-nvidia/libGLX_nvidia.so.0",\n        "api_version" : "1.3.0"\n    }\n}')

# XDG runtime dir
os.makedirs("/tmp/runtime-ue5", exist_ok=True)
os.chmod("/tmp/runtime-ue5", 0o700)

# Create ue5user
subprocess.run(["useradd", "-m", "-s", "/bin/bash", "ue5user"], capture_output=True)

# Copy Claude OAuth credentials from root → ue5user
subprocess.run(["mkdir", "-p", "/home/ue5user/.config"], capture_output=True)
for src in ["/root/.config/claude", "/root/.claude", "/root/.claude.json"]:
    if os.path.exists(src):
        subprocess.run(["cp", "-r", src, "/home/ue5user/.config/" if "config" in src else "/home/ue5user/"],
                       capture_output=True)

# Permissions
subprocess.run(["chown", "-R", "ue5user:ue5user", "/home/ue5user"], capture_output=True)
subprocess.run(["chown", "-R", "ue5user:ue5user", "/tmp/runtime-ue5"], capture_output=True)
subprocess.run(["chmod", "777", "/content"], capture_output=True)
subprocess.run(["mkdir", "-p", "/content/studio_workspace/logs"], capture_output=True)
subprocess.run(["chmod", "-R", "777", "/content/studio_workspace"], capture_output=True)
subprocess.run(["chown", "-R", "ue5user:ue5user", SIMWORLD_DIR], capture_output=True)
subprocess.run(["mkdir", "-p", f"{SIMWORLD_DIR}/gym_citynav/Saved/Logs"], capture_output=True)
subprocess.run(["chown", "-R", "ue5user:ue5user", f"{SIMWORLD_DIR}/gym_citynav/Saved"], capture_output=True)
for log in ["/content/ue.log", "/content/ue_err.log", "/content/studio.log"]:
    open(log, "w").close()
    subprocess.run(["chmod", "666", log], capture_output=True)

# Restore real binary if wrapper was previously installed
if os.path.exists(UE_REAL):
    os.replace(UE_REAL, UE_BINARY)
subprocess.run(["chown", "ue5user:ue5user", UE_BINARY], capture_output=True)
subprocess.run(["chmod", "+x", UE_BINARY], capture_output=True)
print("  Done.")

# --- [4/6] Launch UE as ue5user ---
print("[4/6] Launching SimWorld (UE5)...")
subprocess.run(["pkill", "-9", "-f", "UnrealEditor"], capture_output=True)
subprocess.run(["pkill", "-f", "simworld-studio"], capture_output=True)
time.sleep(3)
subprocess.run(["rm", "-f", f"{SIMWORLD_DIR}/gym_citynav/Saved/Logs/gym_citynav.log"], capture_output=True)

with open("/tmp/launch_ue.sh", "w") as f:
    f.write(f"""#!/bin/bash
export DISPLAY=:99
export CUDA_VISIBLE_DEVICES=0
export VK_ICD_FILENAMES=/usr/share/vulkan/icd.d/nvidia_icd.json
export VK_LOADER_LAYERS_DISABLE='*'
export VK_LAYER_PATH=
export XDG_RUNTIME_DIR=/tmp/runtime-ue5
export LD_LIBRARY_PATH=/usr/lib64-nvidia:$LD_LIBRARY_PATH
export HOME=/home/ue5user

exec {UE_BINARY} {PROJECT_FILE} /Game/Maps/Empty.umap \\
    -MCPPort={MCP_PORT} -RenderOffScreen -Unattended \\
    -NOSPLASH -NOSOUND -ResX=640 -ResY=480 \\
    -FPSMAX=10 -Messaging -graphicsadapter=0 -nopause
""")
subprocess.run(["chmod", "+x", "/tmp/launch_ue.sh"])

ue_proc = subprocess.Popen(
    ["runuser", "-u", "ue5user", "--", "/tmp/launch_ue.sh", "-PixelStreamingIP=127.0.0.1", "-PixelStreamingPort=8586"],
    stdout=open("/content/ue.log", "w"),
    stderr=open("/content/ue_err.log", "w"),
)
print(f"  UE5 started (PID {ue_proc.pid}). Waiting for MCP port {MCP_PORT}...")
print("  This takes 5–15 min on Colab T4. Go grab a coffee ☕")

started = False
for attempt in range(300):  # up to ~25 min
    time.sleep(5)
    alive = subprocess.run(["pgrep", "-f", "UnrealEditor"], capture_output=True).returncode == 0
    if not alive:
        print("\n  ERROR: UnrealEditor died!")
        os.system(f"tail -20 {SIMWORLD_DIR}/gym_citynav/Saved/Logs/gym_citynav.log 2>/dev/null || tail -20 /content/ue.log")
        raise RuntimeError("UnrealEditor crashed.")
    try:
        s = socket.socket(); s.settimeout(2)
        s.connect(("127.0.0.1", MCP_PORT)); s.close()
        started = True; break
    except: pass
    if attempt % 12 == 0 and attempt > 0:
        try:
            last = open(f"{SIMWORLD_DIR}/gym_citynav/Saved/Logs/gym_citynav.log").readlines()[-1].strip()
        except:
            last = "(log not yet available)"
        print(f"  {attempt*5}s: {last[:100]}")

if not started:
    raise RuntimeError("MCP port never opened. Check /content/ue.log")
print(f"  ✓ SimWorld running on port {MCP_PORT}!")

# --- [5/6] Launch Studio as ue5user ---
# Must run as ue5user — studio calls `claude --dangerously-skip-permissions`
# which is blocked for root.
print("[5/6] Launching Studio backend (as ue5user)...")
with open("/tmp/launch_studio.sh", "w") as f:
    f.write(f"""#!/bin/bash
export PATH=/tools/node/bin:$PATH
export DISPLAY=:99
export XDG_RUNTIME_DIR=/tmp/runtime-ue5
export HOME=/home/ue5user

exec simworld-studio start \\
    --mcp-port {MCP_PORT} \\
    --binary {SIMWORLD_DIR} \\
    --port 3002 \\
    --data-dir /content/studio_workspace \\
    --skip-gpu-check
""")
subprocess.run(["chmod", "+x", "/tmp/launch_studio.sh"])

studio_proc = subprocess.Popen(
    ["runuser", "-u", "ue5user", "--", "/tmp/launch_studio.sh"],
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"  Studio started (PID {studio_proc.pid}). Waiting for health check...")

# --- [6/6] Wait for Studio health ---
print("[6/6] Waiting for Studio to be ready...")
ready = False
for i in range(24):
    time.sleep(5)
    if studio_proc.poll() is not None:
        print("  ERROR: Studio exited!")
        os.system("cat /content/studio.log")
        raise RuntimeError("Studio exited unexpectedly.")
    try:
        r = requests.get("http://localhost:3002/api/health", timeout=5)
        if r.json().get("status") == "ok":
            ready = True
            print(f"  ✓ Studio ready! Health: {r.json()}")
            break
    except: pass
    print(f"  {(i+1)*5}s — not ready yet...")

if not ready:
    os.system("cat /content/studio.log")
    raise RuntimeError("Studio backend failed to start.")

print("\n✅ All services launched!")
print(f"   SimWorld MCP : 127.0.0.1:{MCP_PORT}")
print(f"   Studio UI    : http://localhost:3002")
print(f"\n   Tip: Run the next cell to get your public tunnel URL.")

[1/6] Installing rendering dependencies...
  Done.
[2/6] Starting virtual display...
  Virtual display :99 started.
[3/6] Configuring Vulkan, user, and permissions...
  Done.
[4/6] Launching SimWorld (UE5)...
  UE5 started (PID 55522). Waiting for MCP port 55559...
  This takes 5–15 min on Colab T4. Go grab a coffee ☕
  ✓ SimWorld running on port 55559!
[5/6] Launching Studio backend (as ue5user)...
  Studio started (PID 55694). Waiting for health check...
[6/6] Waiting for Studio to be ready...
  ✓ Studio ready! Health: {'status': 'ok', 'ueConnected': True, 'mcpConnected': True, 'pixelStreamingUrl': 'http://127.0.0.1:8585'}

✅ All services launched!
   SimWorld MCP : 127.0.0.1:55559
   Studio UI    : http://localhost:3002

   Tip: Run the next cell to get your public tunnel URL.


In [82]:
"""
Fix live stream: open a second Cloudflare tunnel directly to Cirrus WS port 8586,
then patch the /ue route to use that tunnel URL.
No proxy injection — just a URL swap.
"""
import subprocess, time, re, os, select

INDEX_JS = "/content/studio_workspace/web/server/index.js"

# 1. Start tunnel pointing to Cirrus WS port
print("Opening tunnel to Cirrus (port 8586)...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8586", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

ws_url = None
accumulated = ""
for _ in range(30):
    time.sleep(1)
    try:
        ready, _, _ = select.select([tunnel.stderr], [], [], 0)
        if ready:
            chunk = os.read(tunnel.stderr.fileno(), 8192).decode(errors="replace")
            accumulated += chunk
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", accumulated)
            if match:
                ws_url = match.group(0)
                break
    except Exception:
        pass

if not ws_url:
    print("❌ Tunnel failed to start. Check cloudflared is installed.")
else:
    wss_url = ws_url.replace("https://", "wss://")
    print(f"✓ Cirrus tunnel: {wss_url}")

    # 2. Patch the /ue route — swap the hardcoded ws://hostname:8080 for the real URL
    with open(INDEX_JS, "r") as f:
        src = f.read()

    if "ws://'+location.hostname+':8080'" in src:
        src = src.replace(
            "ws://'+location.hostname+':8080'",
            f"'{wss_url}'"
        )
        with open(INDEX_JS, "w") as f:
            f.write(src)
        print(f"✓ /ue route patched to use {wss_url}")
    elif wss_url in src:
        print("✓ Route already patched")
    else:
        print("⚠ Pattern not found — check index.js /ue route manually")

    # 3. Verify syntax then restart node only (UE keeps running)
    check = subprocess.run(["node", "--check", INDEX_JS], capture_output=True, text=True)
    if check.returncode != 0:
        print("❌ Syntax error:", check.stderr[:200])
    else:
        subprocess.run(["pkill", "-f", "node.*index.js"], capture_output=True)
        time.sleep(2)
        studio_proc = subprocess.Popen(
            ["runuser", "-u", "ue5user", "--", "/tmp/launch_studio.sh"],
            stdout=open("/content/studio.log", "w"),
            stderr=subprocess.STDOUT,
        )
        print(f"✓ Node restarted (PID {studio_proc.pid})")
        print("Hard-refresh browser (Ctrl+Shift+R) then click 'Activate Stream'")

Opening tunnel to Cirrus (port 8586)...
✓ Cirrus tunnel: wss://maintains-timing-ranges-cardiff.trycloudflare.com
⚠ Pattern not found — check index.js /ue route manually
✓ Node restarted (PID 55891)
Hard-refresh browser (Ctrl+Shift+R) then click 'Activate Stream'


## Cell 6: Get Your Browser URL

In [ ]:
"""
Cell 6: Start tunnels for SimWorld Studio.
- UI (port 3002)  → Colab proxy (supports SSE/streaming, fixes chat)
- Cirrus (port 8586) → Cloudflare tunnel (WSS for pixel streaming)
"""
import subprocess, time, re, os, select
from google.colab.output import eval_js

INDEX_JS = "/content/studio_workspace/web/server/index.js"

def get_cloudflare_tunnel(port):
    t = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    accumulated = ""
    for _ in range(30):
        time.sleep(1)
        try:
            ready, _, _ = select.select([t.stderr], [], [], 0)
            if ready:
                chunk = os.read(t.stderr.fileno(), 8192).decode(errors="replace")
                accumulated += chunk
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", accumulated)
                if match:
                    return t, match.group(0)
        except Exception:
            pass
    return t, None

# Kill any existing tunnels
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# UI: use Colab proxy — it supports SSE so chat works properly
print("Getting Colab proxy URL for UI (port 3002)...")
public_url = eval_js("google.colab.kernel.proxyPort(3002)")
print(f"✓ UI:     {public_url}")

# Cirrus: needs Cloudflare tunnel for WSS pixel streaming
print("Starting Cirrus tunnel (port 8586)...")
t2, cirrus_url = get_cloudflare_tunnel(8586)
if not cirrus_url:
    print("❌ Cirrus tunnel failed — live stream won't work but chat will")
    wss_url = None
else:
    wss_url = cirrus_url.replace("https://", "wss://")
    print(f"✓ Cirrus: {wss_url}")

    # Patch /ue route in index.js with the live WSS URL
    with open(INDEX_JS, "r") as f:
        src = f.read()

    new_src = re.sub(
        r"wss://[a-z0-9-]+\.trycloudflare\.com",
        wss_url, src
    )
    if new_src == src:
        # First time — no existing trycloudflare URL yet, use fallback patterns
        new_src = re.sub(
            r"(wss?://[^'\"]*(?:3002/ps-ws|8080|8585|8586)[^'\"]*)",
            wss_url, src
        )
    if new_src == src:
        new_src = src.replace(
            "ws://'+location.hostname+':3002/ps-ws'",
            f"'{wss_url}'"
        )

    with open(INDEX_JS, "w") as f:
        f.write(new_src)
    print(f"✓ /ue route patched → {wss_url}")

# Restart node to pick up the new WSS URL
subprocess.run(["pkill", "-f", "node.*index.js"], capture_output=True)
subprocess.run(["pkill", "-f", "simworld-studio"], capture_output=True)
time.sleep(2)
proc = subprocess.Popen(
    ["runuser", "-u", "ue5user", "--", "/tmp/launch_studio.sh"],
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(8)
print(f"✓ Studio restarted (PID {proc.pid})")

print(f"""
{'='*60}
  SimWorld Studio is live!

  Open in browser: {public_url}
{'='*60}

Chat uses Colab proxy  → SSE streaming works ✓
Stream uses Cloudflare → pixel streaming works ✓
Re-run this cell each session (tunnel URLs change on restart).
""")

Getting Colab proxy URL for UI (port 3002)...


In [91]:
# Also check: is the SSE connection failing because Cloudflare 
# strips the tunnel's Authorization header?
# Test /api/chat directly from Colab (simulates what browser does)
import requests, time

print("Testing SSE stream from /api/chat...")
try:
    with requests.post(
        "http://localhost:3002/api/chat",
        json={"message": "say hi in one word"},
        stream=True,
        timeout=30,
        headers={"Accept": "text/event-stream"}
    ) as r:
        print(f"Status: {r.status_code}")
        print(f"Headers: {dict(r.headers)}")
        for i, chunk in enumerate(r.iter_lines()):
            if chunk:
                print(f"Chunk {i}: {chunk[:200]}")
            if i > 10:
                break
except Exception as e:
    print(f"Error: {e}")

Testing SSE stream from /api/chat...
Status: 200
Headers: {'X-Powered-By': 'Express', 'Access-Control-Allow-Origin': '*', 'Content-Type': 'text/event-stream', 'Cache-Control': 'no-cache', 'Connection': 'keep-alive', 'X-Accel-Buffering': 'no', 'Date': 'Wed, 01 Apr 2026 07:16:28 GMT', 'Transfer-Encoding': 'chunked'}
Chunk 0: b'event: system'
Chunk 1: b'data: {"sessionId":"ba20bc2f-35e9-45e8-a6e7-0f2568230392","mcpServers":[{"name":"simworld","status":"connected"},{"name":"claude.ai Google Calendar","status":"needs-auth"},{"name":"claude.ai Gmail","s'
Chunk 3: b'event: text'
Chunk 4: b'data: {"delta":"Hi!"}'
Chunk 6: b'event: done'
Chunk 7: b'data: {"sessionId":"ba20bc2f-35e9-45e8-a6e7-0f2568230392","isError":false,"costUsd":0.0263736,"latestScreenshot":"/api/screenshot/file?path=%2Fcontent%2Fstudio_workspace%2Ftmp%2Fscreens%2Fscreenshot_1'


In [68]:
# 1. Is the Claude/agent process still alive?
!ps aux | grep -E "claude|node.*studio" | grep -v grep

root        3196  0.1  1.8 11920424 239660 pts/1 Sl+  03:41   0:17 claude


## Cell 7: Verify Everything Works

Run this cell to confirm all services are healthy before using the Studio.

In [7]:
import os

# claude is installed at /tools/node/bin — add to PATH so all subprocess calls find it
os.environ["PATH"] = "/tools/node/bin:" + os.environ["PATH"]

# Verify
import subprocess
r = subprocess.run(["claude", "auth", "status"], capture_output=True, text=True)
print(r.stdout)

{
  "loggedIn": true,
  "authMethod": "claude.ai",
  "apiProvider": "firstParty",
  "email": "kah044@ucsd.edu",
  "orgId": "cab19934-68b8-4f26-9dfc-4d920a53e5c8",
  "orgName": "kah044@ucsd.edu's Organization",
  "subscriptionType": "pro"
}



In [8]:
"""Automated verification — all checks must pass before using the Studio."""
import requests, socket, subprocess, os

print("Running verification checks...\n")

results = {}

# 1. GPU
r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True)
results["GPU"] = (r.returncode == 0, r.stdout.strip() if r.returncode == 0 else "Not found")

# 2. SimWorld binary
ue_bin = "/content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor"
results["SimWorld Binary"] = (os.path.exists(ue_bin), ue_bin if os.path.exists(ue_bin) else "Not found")

# 3. SimWorld TCP (MCP port)
mcp_port = globals().get('MCP_PORT', 55559)
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(("127.0.0.1", mcp_port))
    s.close()
    results[f"SimWorld MCP ({mcp_port})"] = (True, "Connected")
except Exception as e:
    results[f"SimWorld MCP ({mcp_port})"] = (False, str(e))

# 4. Studio Backend
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    results["Studio Backend"] = (r.status_code == 200, f"HTTP {r.status_code}")
except Exception as e:
    results["Studio Backend"] = (False, str(e))

# 5. Skills
try:
    r = requests.get("http://localhost:3002/api/skills", timeout=10)
    data = r.json()
    count = len(data) if isinstance(data, list) else 0
    results["Skills"] = (count > 0, f"{count} skills loaded")
except Exception as e:
    results["Skills"] = (False, str(e))

# 6. Assets
try:
    r = requests.get("http://localhost:3002/api/assets", timeout=10)
    results["Assets"] = (r.status_code == 200, "Catalog loaded")
except Exception as e:
    results["Assets"] = (False, str(e))

# 7. Claude Code CLI
r = subprocess.run(["claude", "--version"], capture_output=True, text=True)
results["Claude Code CLI"] = (r.returncode == 0,
    r.stdout.strip() if r.returncode == 0 else "Not installed")

# 8. Authentication (API key OR Claude Code login)
key_set = bool(os.environ.get("ANTHROPIC_API_KEY", ""))
claude_auth = subprocess.run(["claude", "auth", "status"], capture_output=True, text=True)
# oauth_ok = claude_auth.returncode == 0 and "authenticated" in claude_auth.stdout.lower()
oauth_ok = claude_auth.returncode == 0 and (
    "authenticated" in claude_auth.stdout.lower() or
    '"loggedIn": true' in claude_auth.stdout
)
auth_ok = key_set or oauth_ok
auth_detail = "API key" if key_set else ("Claude Code login" if oauth_ok else "NOT SET — run Cell 4")
results["Authentication"] = (auth_ok, auth_detail)

# 9. Tunnel
tunnel_url = globals().get('public_url', '')
tunnel_ok = bool(tunnel_url) and tunnel_url.startswith("http")
results["Public URL"] = (tunnel_ok, tunnel_url if tunnel_ok else "Not created")

# Print results
all_ok = True
for name, (ok, detail) in results.items():
    icon = "[PASS]" if ok else "[FAIL]"
    if not ok:
        all_ok = False
    print(f"  {icon} {name}: {detail}")

print()
if all_ok:
    print("ALL CHECKS PASSED! SimWorld Studio is ready.")
    if tunnel_url:
        print(f"Open {tunnel_url} in your browser to start building 3D scenes.")
else:
    print("SOME CHECKS FAILED. Common fixes:")
    print("  - SimWorld MCP: wait 30-60s more, then re-run this cell")
    print("  - Authentication: run Cell 4 or use '!claude login'")
    print("  - Tunnel: re-run Cell 6")
    print("  - Studio Backend: !cat /content/studio.log")

Running verification checks...

  [PASS] GPU: Tesla T4
  [PASS] SimWorld Binary: /content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor
  [PASS] SimWorld MCP (55559): Connected
  [PASS] Studio Backend: HTTP 200
  [PASS] Skills: 5 skills loaded
  [PASS] Assets: Catalog loaded
  [PASS] Claude Code CLI: 2.1.89 (Claude Code)
  [PASS] Authentication: Claude Code login
  [PASS] Public URL: https://roulette-compact-sending-statutory.trycloudflare.com

ALL CHECKS PASSED! SimWorld Studio is ready.
Open https://roulette-compact-sending-statutory.trycloudflare.com in your browser to start building 3D scenes.


## Cell 8: Smoke Test (End-to-End)

Sends a test prompt through the full pipeline: **Chat -> Claude -> MCP Tools -> SimWorld**

In [9]:
"""End-to-end smoke test — verifies the full pipeline works."""
import requests, json

print("Sending test prompt through the full pipeline...\n")
print("Prompt: 'Set up the environment with a sunny sky, then spawn one small building'\n")

try:
    response = requests.post("http://localhost:3002/api/chat", json={
        "message": "Set up the environment with a sunny sky, then spawn one small residential building (BP_Building_01) at the origin. Then take a screenshot.",
        "sessionId": None,
        "skills": ["building_placement"]
    }, stream=True, timeout=180)

    tool_calls = []
    text_output = []
    screenshots = []
    errors = []
    current_event = None  # Track SSE event type

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode('utf-8')

        # SSE comment lines (heartbeat)
        if decoded.startswith(':'):
            continue

        # Parse SSE event/data pairs
        if decoded.startswith('event: '):
            current_event = decoded[7:]
            continue
        if not decoded.startswith('data: '):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or 'unknown'

        if event_type == 'text':
            delta = data.get('delta', '')
            text_output.append(delta)
            print(delta, end='', flush=True)
        elif event_type == 'tool_start':
            name = data.get('displayName', data.get('name', '?'))
            tool_calls.append(name)
            print(f"\n  >> Tool: {name}", flush=True)
        elif event_type == 'tool_result':
            result = data.get('result', '')[:200]
            is_error = data.get('isError', False)
            if is_error:
                errors.append(result)
                print(f"\n  >> ERROR: {result}", flush=True)
            else:
                print(f"\n  >> Result: {result[:100]}...", flush=True)
        elif event_type == 'screenshot':
            screenshots.append(data.get('filepath', ''))
            print(f"\n  >> Screenshot captured!", flush=True)
        elif event_type == 'done':
            cost = data.get('costUsd')
            if cost:
                print(f"\n\n  Cost: ${cost:.4f}")

    print(f"\n\n{'='*50}")
    print(f"Smoke Test Results:")
    print(f"  Tool calls:  {len(tool_calls)} ({', '.join(tool_calls)})")
    print(f"  Screenshots: {len(screenshots)}")
    print(f"  Errors:      {len(errors)}")

    if len(tool_calls) > 0 and len(errors) == 0:
        print(f"\n  SMOKE TEST PASSED!")
        print(f"  Full pipeline working: Chat -> Claude -> MCP -> SimWorld")
    elif len(tool_calls) > 0:
        print(f"\n  PARTIAL PASS — tools fired but some errors occurred.")
    else:
        print(f"\n  SMOKE TEST FAILED — no tool calls detected.")
        print(f"  Check: API key set? SimWorld running? Logs: /content/studio.log")

except requests.exceptions.Timeout:
    print("\nSmoke test timed out (180s). Claude may be slow to respond.")
    print("The platform may still work — try using the browser UI.")
except Exception as e:
    print(f"\nSmoke test error: {e}")
    print("Check /content/studio.log for backend errors.")

Sending test prompt through the full pipeline...

Prompt: 'Set up the environment with a sunny sky, then spawn one small building'


  >> Tool: ToolSearch

  >> Result: ...
Starting the workflow — clearing the scene first.
  >> Tool: delete_all_spawned

  >> Result: {
  "result": {
    "success": true,
    "python_logs": [
      "[132] Deleted 36 actors: ['Arena_En...
Scene cleared. Now setting up the sunny environment.
  >> Tool: setup_environment

  >> Result: {
  "status": "success",
  "message": "Environment set up: sun (noon), sky atmosphere, sky light, fo...
Environment ready. Spawning BP_Building_01 at the origin.
  >> Tool: spawn_blueprint_actor

  >> Result: {
  "status": "success",
  "result": {
    "name": "House_01",
    "class": "BP_Building_01_C",
    ...
Building placed. Taking the screenshot.
  >> Tool: take_screenshot

  >> Screenshot captured!

  >> Result: {
  "status": "success",
  "result": {
    "filepath": "/content/studio_workspace/tmp/screens/screen...
Done. He

In [79]:
# Check if the backend process is still alive
!ps aux | grep -E "simworld|node" | grep -v grep

root           7  0.1  0.5 1309176 68840 ?       Sl   03:28   0:14 /tools/node/bin/node /datalab/web/app.js


---

## Troubleshooting

| Issue | Fix |
|---|---|
| "No GPU detected" | Runtime -> Change runtime type -> GPU (T4) |
| SimWorld TCP fails | Wait 60s more, re-run Cell 7 |
| Studio backend fails | Check: `!cat /content/studio.log` |
| Tunnel fails | Re-run Cell 6, or use Colab proxy |
| Claude errors | Verify API key in Cell 4 |
| Session expires | Colab free tier = 12h max. Re-run all cells. |

### View Logs
```python
!tail -50 /content/ue.log      # SimWorld logs
!tail -50 /content/studio.log    # Studio backend logs
```

## Scene Verifier

### 1. Setup & Dependencies

In [ ]:
# Install dependencies (skip if already installed from Studio setup)
# !pip install -q anthropic Pillow numpy scipy matplotlib

import json
import math
import re
import os
import time
import base64
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from io import BytesIO

import requests
import numpy as np
from PIL import Image, ImageDraw
import anthropic

# ── Studio API Configuration ──
# These match the Studio launched in earlier cells
STUDIO_BASE_URL = "http://localhost:3002"
STUDIO_CHAT_URL = f"{STUDIO_BASE_URL}/api/chat"
STUDIO_HEALTH_URL = f"{STUDIO_BASE_URL}/api/health"
STUDIO_SKILLS_URL = f"{STUDIO_BASE_URL}/api/skills"
STUDIO_ASSETS_URL = f"{STUDIO_BASE_URL}/api/assets"

# Anthropic client for VLM calls
vlm_client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY from env

# Verify Studio is reachable
try:
    r = requests.get(STUDIO_HEALTH_URL, timeout=10)
    r.raise_for_status()
    print(f"[OK] Studio backend healthy: {r.json()}")
except Exception as e:
    print(f"[WARN] Studio backend not reachable: {e}")
    print("  Make sure Cells 5-7 ran successfully before continuing.")

print("Dependencies loaded.")

### 2. Data Structures

In [ ]:
@dataclass
class SceneActor:
    """A single object in the scene graph, populated from tool call results."""
    actor_type: str          # e.g. 'BP_Building_Residential_01'
    category: str            # coarse: 'building', 'vegetation', 'vehicle', 'prop'
    position: tuple          # (x, y, z) in UE5 world units (cm)
    rotation: tuple          # (pitch, yaw, roll) in degrees
    scale: tuple             # (sx, sy, sz)
    bbox_extent: tuple       # half-extents (ex, ey, ez) in cm

    @property
    def bbox_min(self):
        return tuple(p - e for p, e in zip(self.position, self.bbox_extent))

    @property
    def bbox_max(self):
        return tuple(p + e for p, e in zip(self.position, self.bbox_extent))


@dataclass
class SceneObservation:
    """All data captured from a real generated scene."""
    prompt: str
    scene_graph: list                                    # list of SceneActor
    rgb_views: dict = field(default_factory=dict)        # camera_name -> PIL Image
    depth_views: dict = field(default_factory=dict)      # camera_name -> np.ndarray
    seg_views: dict = field(default_factory=dict)        # camera_name -> PIL Image
    reference_rgb: Optional[Image.Image] = None
    reference_depth: Optional[np.ndarray] = None
    reference_seg: Optional[np.ndarray] = None
    raw_tool_calls: list = field(default_factory=list)   # raw SSE tool data for debugging
    session_id: Optional[str] = None


@dataclass
class VerifierResult:
    """Full verifier output."""
    composite_score: float
    tier1: dict              # {semantic, physical, structural} scores + details
    tier2: Optional[dict]    # {shape, depth, perceptual} or None
    diagnostics: list        # list of diagnostic strings

    def summary(self):
        lines = [f"Composite Score: {self.composite_score:.2f}"]
        lines.append(f"  Tier 1: semantic={self.tier1['semantic']['score']:.2f}, "
                     f"physical={self.tier1['physical']['score']:.2f}, "
                     f"structural={self.tier1['structural']['score']:.2f}")
        if self.tier2:
            lines.append(f"  Tier 2: shape={self.tier2['shape']['score']:.2f}, "
                         f"depth={self.tier2['depth']['score']:.2f}, "
                         f"perceptual={self.tier2['perceptual']['score']:.2f}")
        lines.append("Diagnostics:")
        for d in self.diagnostics:
            lines.append(f"  {d}")
        return "\n".join(lines)

print("Data structures defined.")

### 3. Observation Capture — Studio API Integration

In [ ]:
def send_chat_prompt(prompt: str, skills: list = None, session_id: str = None,
                     timeout: int = 300) -> dict:
    """
    Send a prompt to the Studio chat API and parse the full SSE response.

    Returns dict with keys:
      - tool_calls: list of {name, input, result, is_error}
      - text_chunks: list of str (Claude's text output)
      - screenshots: list of str (filepaths on server)
      - session_id: str
      - cost_usd: float or None
      - raw_events: list of (event_type, data) for debugging
    """
    payload = {
        "message": prompt,
        "sessionId": session_id,
    }
    if skills:
        payload["skills"] = skills

    response = requests.post(
        STUDIO_CHAT_URL,
        json=payload,
        stream=True,
        timeout=timeout,
    )
    response.raise_for_status()

    result = {
        "tool_calls": [],
        "text_chunks": [],
        "screenshots": [],
        "session_id": session_id,
        "cost_usd": None,
        "raw_events": [],
    }

    current_event = None
    current_tool = None

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8")

        if decoded.startswith(":"):
            continue
        if decoded.startswith("event: "):
            current_event = decoded[7:]
            continue
        if not decoded.startswith("data: "):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or "unknown"
        result["raw_events"].append((event_type, data))

        if event_type == "text":
            delta = data.get("delta", "")
            result["text_chunks"].append(delta)

        elif event_type == "tool_start":
            current_tool = {
                "name": data.get("displayName", data.get("name", "?")),
                "input": data.get("input", {}),
                "result": None,
                "is_error": False,
            }

        elif event_type == "tool_result":
            if current_tool:
                current_tool["result"] = data.get("result", "")
                current_tool["is_error"] = data.get("isError", False)
                result["tool_calls"].append(current_tool)
                current_tool = None

        elif event_type == "screenshot":
            filepath = data.get("filepath", "")
            if filepath:
                result["screenshots"].append(filepath)

        elif event_type == "done":
            cost = data.get("costUsd")
            if cost:
                result["cost_usd"] = cost
            sid = data.get("sessionId")
            if sid:
                result["session_id"] = sid

    return result


print("Chat API integration defined.")

In [ ]:
# ── Category inference from actor type names ──
CATEGORY_PATTERNS = {
    "building": ["building", "house", "office", "shop", "store", "commercial",
                 "residential", "apartment", "tower", "warehouse", "factory"],
    "vegetation": ["tree", "bush", "shrub", "hedge", "flower", "plant",
                   "grass", "oak", "pine", "maple", "palm", "vegetation"],
    "vehicle": ["vehicle", "car", "sedan", "truck", "bus", "van", "suv",
                "motorcycle", "bike", "taxi"],
    "road": ["road", "street", "sidewalk", "crosswalk", "intersection",
             "highway", "path", "lane"],
    "prop": ["bench", "lamp", "light", "sign", "mailbox", "hydrant",
             "fence", "wall", "barrier", "pole", "bin", "trash", "dumpster"],
}

# ── Default bbox extents when not provided by the engine ──
DEFAULT_BBOX = {
    "building":   (500, 500, 800),
    "vegetation": (200, 200, 400),
    "vehicle":    (250, 120, 80),
    "road":       (500, 100, 10),
    "prop":       (50, 50, 100),
}


def infer_category(actor_type: str) -> str:
    """Infer coarse category from the blueprint/actor type name."""
    name_lower = actor_type.lower()
    for category, keywords in CATEGORY_PATTERNS.items():
        if any(kw in name_lower for kw in keywords):
            return category
    return "prop"


def extract_scene_graph_from_tool_calls(tool_calls: list) -> List[SceneActor]:
    """
    Parse tool call results from the SSE stream to reconstruct the scene graph.

    The Studio agent uses MCP tools like:
      - spawn_actor / place_building / add_vegetation
      - set_actor_transform / move_actor
      - get_scene_actors (returns full actor list)

    We look for get_scene_actors first (authoritative), then fall back to
    accumulating spawn calls.
    """
    actors = []

    # ── Strategy 1: Look for a get_scene_actors / list_actors response ──
    for tc in tool_calls:
        name_lower = tc["name"].lower()
        if any(kw in name_lower for kw in ["get_scene", "list_actor", "scene_graph",
                                           "get_actors", "query_scene"]):
            try:
                result_data = tc["result"]
                if isinstance(result_data, str):
                    result_data = json.loads(result_data)

                actor_list = result_data
                if isinstance(result_data, dict):
                    # Handle {actors: [...]} or {result: [...]}
                    actor_list = (result_data.get("actors")
                                 or result_data.get("result")
                                 or result_data.get("data", []))

                if isinstance(actor_list, list) and len(actor_list) > 0:
                    for a in actor_list:
                        if isinstance(a, dict):
                            actor_type = a.get("type", a.get("actor_type",
                                          a.get("blueprint", a.get("name", "Unknown"))))
                            category = a.get("category", infer_category(actor_type))

                            pos = a.get("position", a.get("location", (0, 0, 0)))
                            if isinstance(pos, dict):
                                pos = (pos.get("x", 0), pos.get("y", 0), pos.get("z", 0))
                            elif isinstance(pos, list):
                                pos = tuple(pos)

                            rot = a.get("rotation", (0, 0, 0))
                            if isinstance(rot, dict):
                                rot = (rot.get("pitch", 0), rot.get("yaw", 0), rot.get("roll", 0))
                            elif isinstance(rot, list):
                                rot = tuple(rot)

                            scale = a.get("scale", (1, 1, 1))
                            if isinstance(scale, dict):
                                scale = (scale.get("x", 1), scale.get("y", 1), scale.get("z", 1))
                            elif isinstance(scale, list):
                                scale = tuple(scale)

                            bbox = a.get("bbox_extent", a.get("extent",
                                         a.get("bounds", DEFAULT_BBOX.get(category, (100, 100, 100)))))
                            if isinstance(bbox, dict):
                                bbox = (bbox.get("x", 100), bbox.get("y", 100), bbox.get("z", 100))
                            elif isinstance(bbox, list):
                                bbox = tuple(bbox)

                            actors.append(SceneActor(
                                actor_type=str(actor_type),
                                category=category,
                                position=tuple(float(v) for v in pos),
                                rotation=tuple(float(v) for v in rot),
                                scale=tuple(float(v) for v in scale),
                                bbox_extent=tuple(float(v) for v in bbox),
                            ))

                    if actors:
                        print(f"  [scene_graph] Extracted {len(actors)} actors from '{tc['name']}'")
                        return actors
            except (json.JSONDecodeError, TypeError, KeyError) as e:
                print(f"  [scene_graph] Failed to parse '{tc['name']}': {e}")

    # ── Strategy 2: Accumulate from individual spawn/place tool calls ──
    for tc in tool_calls:
        if tc["is_error"]:
            continue
        name_lower = tc["name"].lower()
        if not any(kw in name_lower for kw in ["spawn", "place", "add", "create"]):
            continue

        try:
            inp = tc["input"]
            if isinstance(inp, str):
                inp = json.loads(inp)

            actor_type = (inp.get("actor_type") or inp.get("blueprint")
                          or inp.get("asset") or inp.get("type") or tc["name"])
            category = infer_category(str(actor_type))

            pos = inp.get("position", inp.get("location", (0, 0, 0)))
            if isinstance(pos, dict):
                pos = (pos.get("x", 0), pos.get("y", 0), pos.get("z", 0))
            elif isinstance(pos, list):
                pos = tuple(pos)

            rot = inp.get("rotation", (0, 0, 0))
            if isinstance(rot, dict):
                rot = (rot.get("pitch", 0), rot.get("yaw", 0), rot.get("roll", 0))
            elif isinstance(rot, list):
                rot = tuple(rot)

            scale = inp.get("scale", (1, 1, 1))
            if isinstance(scale, dict):
                scale = (scale.get("x", 1), scale.get("y", 1), scale.get("z", 1))
            elif isinstance(scale, list):
                scale = tuple(scale)

            bbox = DEFAULT_BBOX.get(category, (100, 100, 100))

            actors.append(SceneActor(
                actor_type=str(actor_type),
                category=category,
                position=tuple(float(v) for v in pos),
                rotation=tuple(float(v) for v in rot),
                scale=tuple(float(v) for v in scale),
                bbox_extent=bbox,
            ))
        except (json.JSONDecodeError, TypeError, KeyError, ValueError) as e:
            print(f"  [scene_graph] Skipping tool call '{tc['name']}': {e}")

    if actors:
        print(f"  [scene_graph] Accumulated {len(actors)} actors from spawn/place calls")
    else:
        print("  [scene_graph] WARNING: No actors extracted from tool calls")

    return actors


print("Scene graph extraction defined.")

In [ ]:
def fetch_screenshot_as_pil(filepath: str) -> Optional[Image.Image]:
    """
    Fetch a screenshot from the Studio server.
    The filepath comes from the SSE 'screenshot' event.
    """
    # Studio may serve screenshots at /api/screenshot/<path> or as static files
    for url_pattern in [
        f"{STUDIO_BASE_URL}/api/screenshot?path={filepath}",
        f"{STUDIO_BASE_URL}/screenshots/{os.path.basename(filepath)}",
        f"{STUDIO_BASE_URL}{filepath}" if filepath.startswith("/") else None,
    ]:
        if url_pattern is None:
            continue
        try:
            r = requests.get(url_pattern, timeout=30)
            if r.status_code == 200 and len(r.content) > 1000:
                return Image.open(BytesIO(r.content)).convert("RGB")
        except Exception:
            continue

    # If the filepath is a local path on the Colab filesystem
    if os.path.exists(filepath):
        return Image.open(filepath).convert("RGB")

    # Check workspace
    workspace_path = f"/content/studio_workspace/{filepath}"
    if os.path.exists(workspace_path):
        return Image.open(workspace_path).convert("RGB")

    print(f"  [screenshot] Could not fetch: {filepath}")
    return None


def request_screenshot(session_id: str = None) -> dict:
    """Ask the agent to take a screenshot via a follow-up chat message."""
    return send_chat_prompt(
        "Take a screenshot of the current scene from a bird's eye view.",
        session_id=session_id,
        timeout=120,
    )


def capture_observation(prompt: str, skills: list = None,
                        request_additional_views: bool = True) -> SceneObservation:
    """
    Full observation capture pipeline:
      1. Send generation prompt to Studio
      2. Parse tool calls to build scene graph
      3. Collect screenshots from the response
      4. Optionally request scene graph query + extra views

    Returns a populated SceneObservation ready for verification.
    """
    print(f"[capture] Sending prompt: \"{prompt[:80]}...\"")

    # ── Step 1: Generate the scene ──
    chat_result = send_chat_prompt(prompt, skills=skills)
    session_id = chat_result["session_id"]

    print(f"  Tool calls: {len(chat_result['tool_calls'])}")
    print(f"  Screenshots: {len(chat_result['screenshots'])}")
    if chat_result["cost_usd"]:
        print(f"  Cost: ${chat_result['cost_usd']:.4f}")

    # ── Step 2: Extract scene graph from tool calls ──
    scene_graph = extract_scene_graph_from_tool_calls(chat_result["tool_calls"])

    # If no scene graph found from tool calls, ask the agent to query it
    if not scene_graph:
        print("  [capture] No scene graph from generation — requesting explicitly...")
        sg_result = send_chat_prompt(
            "List all actors currently in the scene with their positions, types, "
            "rotations, scales, and bounding box extents. Return as structured data.",
            session_id=session_id,
            timeout=120,
        )
        scene_graph = extract_scene_graph_from_tool_calls(sg_result["tool_calls"])
        chat_result["screenshots"].extend(sg_result["screenshots"])

    # ── Step 3: Collect RGB views ──
    rgb_views = {}
    for i, filepath in enumerate(chat_result["screenshots"]):
        img = fetch_screenshot_as_pil(filepath)
        if img:
            view_name = "bird_eye" if i == 0 else f"view_{i}"
            rgb_views[view_name] = img
            print(f"  [capture] Loaded screenshot: {view_name} ({img.size})")

    # If no screenshots were captured during generation, request one
    if not rgb_views:
        print("  [capture] No screenshots in generation — requesting one...")
        ss_result = request_screenshot(session_id)
        for filepath in ss_result["screenshots"]:
            img = fetch_screenshot_as_pil(filepath)
            if img:
                rgb_views["bird_eye"] = img
                print(f"  [capture] Loaded follow-up screenshot ({img.size})")
                break

    # ── Step 4: Build observation ──
    obs = SceneObservation(
        prompt=prompt,
        scene_graph=scene_graph,
        rgb_views=rgb_views,
        raw_tool_calls=chat_result["tool_calls"],
        session_id=session_id,
    )

    print(f"[capture] Observation complete: {len(obs.scene_graph)} actors, "
          f"{len(obs.rgb_views)} views")
    return obs


print("Observation capture pipeline defined.")

### 4. Tier 1 Metric: Structural Score (Rule-Based)

In [ ]:
GROUND_LEVEL = 0        # z=0 is ground in UE5 default
GROUND_TOLERANCE = 50   # allow 50cm tolerance (placement noise)
SCENE_BOUNDS = 50000    # 500m radius from origin


def check_underground(actors: list) -> list:
    """Check if any actor's bounding box bottom is below ground."""
    issues = []
    for a in actors:
        z_bottom = a.position[2] - a.bbox_extent[2]
        if z_bottom < GROUND_LEVEL - GROUND_TOLERANCE:
            issues.append(f"[FAIL] {a.actor_type} at {a.position} is underground "
                          f"(bottom z={z_bottom:.0f}, ground={GROUND_LEVEL})")
    return issues


def check_floating(actors: list) -> list:
    """Check if any actor is floating above ground with no support."""
    issues = []
    for a in actors:
        z_bottom = a.position[2] - a.bbox_extent[2]
        float_threshold = 100 if a.category == 'building' else 200
        if z_bottom > GROUND_LEVEL + float_threshold:
            issues.append(f"[FAIL] {a.actor_type} at {a.position} is floating "
                          f"(bottom z={z_bottom:.0f}, "
                          f"{z_bottom - GROUND_LEVEL:.0f}cm above ground)")
    return issues


def check_collisions(actors: list) -> list:
    """Check for bounding box overlaps between actors."""
    issues = []
    for i in range(len(actors)):
        for j in range(i + 1, len(actors)):
            a, b = actors[i], actors[j]
            overlap = all(
                a.bbox_min[k] < b.bbox_max[k] and b.bbox_min[k] < a.bbox_max[k]
                for k in range(3)
            )
            if overlap:
                issues.append(f"[FAIL] Collision: {a.actor_type} at {a.position} "
                              f"overlaps with {b.actor_type} at {b.position}")
    return issues


def check_out_of_bounds(actors: list) -> list:
    """Check if any actor is unreasonably far from origin."""
    issues = []
    for a in actors:
        dist = math.sqrt(a.position[0]**2 + a.position[1]**2)
        if dist > SCENE_BOUNDS:
            issues.append(f"[FAIL] {a.actor_type} at {a.position} is out of scene bounds "
                          f"(dist={dist:.0f}, limit={SCENE_BOUNDS})")
    return issues


def compute_structural_score(scene_graph: list) -> dict:
    """Rule-based structural validity checks on the scene graph."""
    if not scene_graph:
        return {'score': 0.0, 'details': ['[FAIL] Scene is empty — no actors'],
                'checks_passed': 0, 'checks_total': 1}

    all_issues = []
    all_issues.extend(check_underground(scene_graph))
    all_issues.extend(check_floating(scene_graph))
    all_issues.extend(check_collisions(scene_graph))
    all_issues.extend(check_out_of_bounds(scene_graph))

    n = len(scene_graph)
    total_checks = 3 * n + (n * (n - 1)) // 2
    failed = len(all_issues)
    passed = total_checks - failed

    details = all_issues.copy()
    if not check_underground(scene_graph):
        details.append('[PASS] No underground objects')
    if not check_floating(scene_graph):
        details.append('[PASS] No floating objects')
    if not check_collisions(scene_graph):
        details.append('[PASS] No collisions detected')
    if not check_out_of_bounds(scene_graph):
        details.append('[PASS] All objects within scene bounds')

    score = max(0.0, passed / total_checks) if total_checks > 0 else 1.0

    return {
        'score': score,
        'details': details,
        'checks_passed': passed,
        'checks_total': total_checks,
    }


print("Structural score defined.")

### 5. Tier 1 Metric: Semantic Alignment Score (VLM + Regex Fallback)

In [ ]:
PROMPT_PARSER_SYSTEM = """\
You are a requirement extractor for 3D scene generation.
Given a user prompt, extract structured requirements as JSON.

Output format (JSON only, no markdown, no explanation):
{
  "objects": [
    {"category": "building", "count": 4, "attributes": ["residential"], "is_minimum": false},
    {"category": "vegetation", "count": 3, "attributes": ["tree"], "is_minimum": true}
  ],
  "spatial": [
    {"description": "houses arranged in a neighborhood", "check": "layout"}
  ],
  "attributes": [
    {"description": "mixed commercial and residential", "check": "mixed_building_types"}
  ]
}

Rules:
- category must be one of: building, vegetation, vehicle, road, prop
- Plurals without counts imply is_minimum=true with count >= 3
- "a" or "one" -> count = 1, is_minimum = false
- Extract ALL implicit requirements ("park" implies vegetation + open space)
- "street" or "road" implies a spatial layout check
- Only output valid JSON, no other text.
"""


def vlm_parse_prompt(prompt: str) -> list:
    """
    Use Claude to extract structured requirements from a prompt.
    Falls back to regex parsing if the API call fails.
    """
    try:
        response = vlm_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            system=PROMPT_PARSER_SYSTEM,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text.strip()
        # Strip markdown fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        parsed = json.loads(raw)

        requirements = []
        for obj in parsed.get("objects", []):
            requirements.append({
                'type': 'object_count',
                'description': f"Requires {obj['count']} {obj['category']}(s)",
                'expected_category': obj['category'],
                'expected_count': obj['count'],
                'is_minimum': obj.get('is_minimum', False),
            })
        for attr in parsed.get("attributes", []):
            requirements.append({
                'type': 'attribute',
                'description': attr['description'],
                'check': attr.get('check', ''),
            })
        for sp in parsed.get("spatial", []):
            requirements.append({
                'type': 'attribute',
                'description': sp['description'],
                'check': sp.get('check', 'has_street'),
            })
        print(f"  [semantic] VLM extracted {len(requirements)} requirements")
        return requirements

    except Exception as e:
        print(f"  [semantic] VLM parsing failed ({e}), falling back to regex")
        return regex_parse_prompt(prompt)


def regex_parse_prompt(prompt: str) -> list:
    """Regex fallback for prompt parsing (same as original mock version)."""
    prompt_lower = prompt.lower()
    requirements = []

    count_patterns = [
        (r'(\d+)\s+(house|building|home|residence)', 'building'),
        (r'(\d+)\s+(tree|oak|pine|maple)', 'vegetation'),
        (r'(\d+)\s+(car|vehicle|sedan|truck)', 'vehicle'),
        (r'\b(a|one|1)\s+(car|vehicle|sedan)', 'vehicle'),
    ]
    for pattern, category in count_patterns:
        match = re.search(pattern, prompt_lower)
        if match:
            count_str = match.group(1)
            count = 1 if count_str in ('a', 'one') else int(count_str)
            requirements.append({
                'type': 'object_count',
                'description': f'Requires {count} {category}(s)',
                'expected_category': category,
                'expected_count': count,
            })

    plural_patterns = [
        (r'\btrees\b', 'vegetation', 3),
        (r'\bbuildings\b', 'building', 2),
        (r'\bhouses\b', 'building', 2),
        (r'\bcars\b', 'vehicle', 2),
    ]
    for pattern, category, min_count in plural_patterns:
        if re.search(pattern, prompt_lower):
            existing = [r for r in requirements if r.get('expected_category') == category]
            if not existing:
                requirements.append({
                    'type': 'object_count',
                    'description': f'Requires >= {min_count} {category}(s) (plural in prompt)',
                    'expected_category': category,
                    'expected_count': min_count,
                    'is_minimum': True,
                })

    if 'commercial' in prompt_lower and 'residential' in prompt_lower:
        requirements.append({
            'type': 'attribute',
            'description': 'Requires BOTH commercial and residential building types',
            'check': 'mixed_building_types',
        })
    if 'park' in prompt_lower:
        requirements.append({
            'type': 'attribute',
            'description': 'Requires a park area (open space with vegetation)',
            'check': 'has_park',
        })
    if 'street' in prompt_lower or 'road' in prompt_lower:
        requirements.append({
            'type': 'attribute',
            'description': 'Requires a street/road layout',
            'check': 'has_street',
        })

    return requirements


print("Prompt parser defined (VLM primary + regex fallback).")

In [ ]:
def check_requirement(req: dict, scene_graph: list) -> tuple:
    """
    Check a single requirement against the scene graph.
    Returns (score: float 0-1, diagnostic: str)
    """
    if req['type'] == 'object_count':
        category = req['expected_category']
        expected = req['expected_count']
        actual = sum(1 for a in scene_graph if a.category == category)
        is_min = req.get('is_minimum', False)

        if is_min:
            if actual >= expected:
                return 1.0, f'[PASS] {actual} {category}(s) found (>= {expected} expected)'
            else:
                return actual / expected, f'[PARTIAL] {actual} {category}(s) found (>= {expected} expected)'
        else:
            if actual == expected:
                return 1.0, f'[PASS] {actual} {category}(s) found ({expected} expected)'
            elif actual > 0:
                return 1.0 - abs(actual - expected) / max(actual, expected), \
                       f'[PARTIAL] {actual} {category}(s) found ({expected} expected)'
            else:
                return 0.0, f'[FAIL] No {category}(s) found ({expected} expected)'

    elif req['type'] == 'attribute':
        check = req.get('check', '')

        if check == 'mixed_building_types':
            types = set(a.actor_type for a in scene_graph if a.category == 'building')
            has_commercial = any('commercial' in t.lower() for t in types)
            has_residential = any('residential' in t.lower() for t in types)
            if has_commercial and has_residential:
                return 1.0, '[PASS] Both commercial and residential building types present'
            elif has_commercial or has_residential:
                return 0.3, '[FAIL] Only one building type present (need both commercial + residential)'
            else:
                return 0.0, '[FAIL] No recognized building types'

        elif check == 'has_park':
            veg_count = sum(1 for a in scene_graph if a.category == 'vegetation')
            if veg_count >= 2:
                return 1.0, f'[PASS] Park area detected ({veg_count} vegetation actors)'
            elif veg_count > 0:
                return 0.5, f'[PARTIAL] Some vegetation ({veg_count}) but minimal park'
            else:
                return 0.0, '[FAIL] No vegetation found for park'

        elif check == 'has_street':
            buildings = [a for a in scene_graph if a.category == 'building']
            if len(buildings) >= 2:
                y_coords = [b.position[1] for b in buildings]
                y_spread = max(y_coords) - min(y_coords)
                x_spread = max(b.position[0] for b in buildings) - min(b.position[0] for b in buildings)
                if y_spread < 500 or x_spread < 500:
                    return 0.8, '[PASS] Buildings arranged along a line (street-like layout)'
                else:
                    return 0.5, '[PARTIAL] Buildings placed but not clearly along a street'
            return 0.3, '[PARTIAL] Not enough buildings to form a street layout'

    return 0.5, f'[INFO] Could not evaluate requirement: {req["description"]}'


def compute_semantic_score(obs: SceneObservation) -> dict:
    """
    Compute semantic alignment score.
    Uses VLM to parse the prompt; falls back to regex if VLM fails.
    """
    requirements = vlm_parse_prompt(obs.prompt)

    if not requirements:
        return {'score': 0.5, 'details': ['[INFO] No extractable requirements from prompt'],
                'requirements': []}

    scores = []
    details = []
    for req in requirements:
        score, diag = check_requirement(req, obs.scene_graph)
        scores.append(score)
        details.append(diag)

    return {
        'score': sum(scores) / len(scores),
        'details': details,
        'requirements': requirements,
    }


print("Semantic score defined.")

### 6. Tier 1 Metric: Physical Plausibility Score (VLM + Scene Graph)

In [ ]:
PHYSICS_VLM_PROMPT = """\
You are evaluating the physical plausibility of a 3D-rendered urban scene.
Examine the provided image(s) and answer each question below.

For each question, respond with a JSON object. Use exactly this format:
{"floating": {"score": 1.0, "reason": "..."},
 "clipping": {"score": 1.0, "reason": "..."},
 "scale":    {"score": 1.0, "reason": "..."},
 "orientation": {"score": 1.0, "reason": "..."},
 "arrangement": {"score": 1.0, "reason": "..."}}

Scoring: 1.0 = pass, 0.5 = minor issues, 0.0 = clear violation.

Questions:
1. FLOATING: Are any objects floating above the ground with no visible support?
2. CLIPPING: Are any objects intersecting or clipping through each other?
3. SCALE: Are object scales consistent (no giant mailbox next to tiny house)?
4. ORIENTATION: Are objects oriented correctly (buildings upright, not tilted)?
5. ARRANGEMENT: Does the overall scene look like a plausible real-world arrangement?

Output ONLY valid JSON, no markdown fences, no explanation.
"""


def vlm_physical_check(images: List[Image.Image]) -> Optional[dict]:
    """Use Claude vision to evaluate physical plausibility from rendered images."""
    if not images:
        return None

    try:
        content = []
        for img in images[:3]:  # max 3 views to keep token cost down
            buf = BytesIO()
            img.save(buf, format='PNG')
            content.append({
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": base64.standard_b64encode(buf.getvalue()).decode(),
                },
            })
        content.append({"type": "text", "text": PHYSICS_VLM_PROMPT})

        response = vlm_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            messages=[{"role": "user", "content": content}],
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  [physical] VLM check failed: {e}")
        return None


def compute_physical_score_from_graph(scene_graph: list) -> dict:
    """Scene-graph-based physical plausibility checks (always available)."""
    checks = {}
    details = []

    # 1. Floating
    floating = [a for a in scene_graph
                if (a.position[2] - a.bbox_extent[2]) > GROUND_LEVEL + 100]
    if not floating:
        checks['floating'] = 1.0
        details.append('[PASS] No floating objects')
    else:
        checks['floating'] = max(0.0, 1.0 - len(floating) / len(scene_graph))
        for f in floating:
            details.append(f'[FAIL] {f.actor_type} floating '
                           f'{f.position[2] - f.bbox_extent[2] - GROUND_LEVEL:.0f}cm above ground')

    # 2. Clipping
    collision_issues = check_collisions(scene_graph)
    if not collision_issues:
        checks['clipping'] = 1.0
        details.append('[PASS] No object clipping detected')
    else:
        checks['clipping'] = max(0.0, 1.0 - len(collision_issues) * 0.3)
        details.extend(collision_issues)

    # 3. Scale consistency
    buildings = [a for a in scene_graph if a.category == 'building']
    if len(buildings) >= 2:
        heights = [a.bbox_extent[2] * 2 for a in buildings]
        height_ratio = max(heights) / min(heights) if min(heights) > 0 else 999
        if height_ratio < 3.0:
            checks['scale'] = 1.0
            details.append(f'[PASS] Building scale consistent (height ratio {height_ratio:.1f}x)')
        elif height_ratio < 5.0:
            checks['scale'] = 0.5
            details.append(f'[PARTIAL] Building scale slightly off (height ratio {height_ratio:.1f}x)')
        else:
            checks['scale'] = 0.0
            details.append(f'[FAIL] Building scale inconsistent (height ratio {height_ratio:.1f}x)')
    else:
        checks['scale'] = 1.0
        details.append('[PASS] Scale check N/A (< 2 buildings)')

    # 4. Orientation
    tilted = [a for a in scene_graph if a.category == 'building'
              and (abs(a.rotation[0]) > 10 or abs(a.rotation[2]) > 10)]
    if not tilted:
        checks['orientation'] = 1.0
        details.append('[PASS] All buildings properly oriented')
    else:
        checks['orientation'] = max(0.0, 1.0 - len(tilted) / max(len(buildings), 1))
        for t in tilted:
            details.append(f'[FAIL] {t.actor_type} tilted: pitch={t.rotation[0]}° roll={t.rotation[2]}°')

    # 5. Arrangement
    if len(scene_graph) >= 2:
        positions = [(a.position[0], a.position[1]) for a in scene_graph]
        spread_x = max(p[0] for p in positions) - min(p[0] for p in positions)
        spread_y = max(p[1] for p in positions) - min(p[1] for p in positions)
        total_spread = max(spread_x, spread_y)
        if total_spread > 500:
            checks['arrangement'] = 1.0
            details.append(f'[PASS] Objects spread across {total_spread:.0f}cm')
        elif total_spread > 100:
            checks['arrangement'] = 0.7
            details.append(f'[PARTIAL] Objects clustered ({total_spread:.0f}cm spread)')
        else:
            checks['arrangement'] = 0.3
            details.append(f'[FAIL] All objects stacked together ({total_spread:.0f}cm spread)')
    else:
        checks['arrangement'] = 0.5

    overall = sum(checks.values()) / len(checks) if checks else 0.0

    return {
        'score': overall,
        'sub_scores': checks,
        'details': details,
    }


def compute_physical_score_combined(obs: SceneObservation) -> dict:
    """
    Combined physical score: scene graph checks + VLM visual check.
    Scene graph is authoritative (0.6 weight); VLM adds visual nuance (0.4 weight).
    Falls back to scene-graph-only if no images or VLM fails.
    """
    graph_result = compute_physical_score_from_graph(obs.scene_graph)

    # Try VLM check on available views
    images = list(obs.rgb_views.values())
    vlm_result = vlm_physical_check(images) if images else None

    if vlm_result is None:
        # Fallback: graph-only
        graph_result['details'].append('[INFO] VLM physical check unavailable — graph-only score')
        return graph_result

    # Blend scores: graph (authoritative) + VLM (visual nuance)
    vlm_scores = {k: v.get("score", 0.5) for k, v in vlm_result.items()
                  if isinstance(v, dict)}
    vlm_avg = sum(vlm_scores.values()) / len(vlm_scores) if vlm_scores else 0.5

    GRAPH_WEIGHT = 0.6
    VLM_WEIGHT = 0.4
    blended_score = GRAPH_WEIGHT * graph_result['score'] + VLM_WEIGHT * vlm_avg

    details = graph_result['details'].copy()
    for key, val in vlm_result.items():
        if isinstance(val, dict):
            score_val = val.get("score", "?")
            reason = val.get("reason", "")
            details.append(f'[VLM] {key}: {score_val} — {reason}')

    return {
        'score': blended_score,
        'sub_scores': {**graph_result.get('sub_scores', {}), 'vlm_avg': vlm_avg},
        'details': details,
    }


print("Physical score defined (combined VLM + scene graph).")

### 7. Score Aggregation & Full Verifier

In [ ]:
TIER1_WEIGHTS = {
    'semantic': 0.40,
    'physical': 0.35,
    'structural': 0.25,
}

TIER2_WEIGHTS = {
    'shape': 0.30,
    'depth': 0.35,
    'perceptual': 0.35,
}
TIER1_TIER2_BLEND = 0.6


def run_verifier(obs: SceneObservation) -> VerifierResult:
    """Run the full verification pipeline on a scene observation."""

    # ── Tier 1 ──
    semantic = compute_semantic_score(obs)
    physical = compute_physical_score_combined(obs)
    structural = compute_structural_score(obs.scene_graph)

    tier1 = {
        'semantic': semantic,
        'physical': physical,
        'structural': structural,
    }

    tier1_composite = (
        TIER1_WEIGHTS['semantic'] * semantic['score'] +
        TIER1_WEIGHTS['physical'] * physical['score'] +
        TIER1_WEIGHTS['structural'] * structural['score']
    )

    # ── Tier 2 (activated when reference data is provided) ──
    tier2 = None
    if obs.reference_rgb is not None and obs.reference_depth is not None:
        # Import tier 2 functions if available
        try:
            gen_view = list(obs.rgb_views.values())[0] if obs.rgb_views else None
            if gen_view and obs.reference_seg is not None:
                gen_seg = np.array(gen_view)[:, :, 0]  # simplified
                ref_seg = obs.reference_seg
                shape_result = compute_segmentation_iou(gen_seg, ref_seg)
            else:
                shape_result = {'score': 0.0, 'details': ['No segmentation data']}

            depth_result = compute_depth_score(
                obs.depth_views.get('bird_eye', obs.reference_depth),
                obs.reference_depth,
            )
            tier2 = {
                'shape': shape_result,
                'depth': depth_result,
                'perceptual': {'score': 0.0, 'details': ['CLIP/LPIPS not loaded']},
            }
        except Exception as e:
            print(f"  [verifier] Tier 2 failed: {e}")
            tier2 = None

    # ── Composite ──
    if tier2:
        tier2_composite = (
            TIER2_WEIGHTS['shape'] * tier2['shape']['score'] +
            TIER2_WEIGHTS['depth'] * tier2['depth']['score'] +
            TIER2_WEIGHTS['perceptual'] * tier2['perceptual']['score']
        )
        composite = TIER1_TIER2_BLEND * tier1_composite + (1 - TIER1_TIER2_BLEND) * tier2_composite
    else:
        composite = tier1_composite

    # ── Diagnostics ──
    diagnostics = []
    diagnostics.extend(semantic.get('details', []))
    diagnostics.extend(physical.get('details', []))
    diagnostics.extend(structural.get('details', []))

    return VerifierResult(
        composite_score=composite,
        tier1=tier1,
        tier2=tier2,
        diagnostics=diagnostics,
    )


print("Verifier pipeline defined.")

### 8. Generate & Verify Scenes

In [ ]:
# ── Define test prompts with varying complexity ──
test_prompts = [
    {
        "name": "Example 1: Good neighborhood",
        "prompt": "Build a small neighborhood with 4 houses and a park with trees. "
                  "Then take a screenshot from a bird's eye view.",
        "skills": ["building_placement"],
    },
    {
        "name": "Example 2: Mixed city block",
        "prompt": "Create a city block with mixed commercial and residential buildings. "
                  "Place at least 2 commercial and 2 residential buildings. "
                  "Then take a screenshot.",
        "skills": ["building_placement"],
    },
    {
        "name": "Example 3: Street with vehicles",
        "prompt": "Place 3 houses along a street with a car parked in front. "
                  "Then take a screenshot from a bird's eye view.",
        "skills": ["building_placement"],
    },
]

# ── Generate and verify each scene ──
results = []
for test in test_prompts:
    print("\n" + "=" * 70)
    print(f"GENERATING: {test['name']}")
    print("=" * 70)

    # Capture observation from live Studio
    obs = capture_observation(
        prompt=test["prompt"],
        skills=test.get("skills"),
    )

    # Run verifier
    result = run_verifier(obs)
    results.append((test["name"], obs, result))

    print(f"\n{'-' * 70}")
    print(result.summary())
    print()

print("=" * 70)
print("All examples generated and verified.")

### 9. Visualization

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def visualize_scene_topdown(obs: SceneObservation, title: str = ""):
    """Top-down visualization of scene graph with bounding boxes."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Left: top-down scene graph ──
    ax = axes[0]
    colors = {
        'building': '#4A90D9',
        'vegetation': '#5CB85C',
        'vehicle': '#D9534F',
        'road': '#888888',
        'prop': '#F0AD4E',
    }

    for actor in obs.scene_graph:
        x, y = actor.position[0], actor.position[1]
        ex, ey = actor.bbox_extent[0], actor.bbox_extent[1]
        color = colors.get(actor.category, '#999999')

        rect = mpatches.FancyBboxPatch(
            (x - ex, y - ey), ex * 2, ey * 2,
            boxstyle="round,pad=20",
            facecolor=color, edgecolor='black', alpha=0.6, linewidth=1.5
        )
        ax.add_patch(rect)

        short_name = actor.actor_type.replace('BP_', '').replace('_', ' ')
        z_note = ''
        if actor.position[2] > 100:
            z_note = f'\n⚠ z={actor.position[2]}'
        elif actor.position[2] < -50:
            z_note = f'\n⚠ UNDERGROUND'
        ax.text(x, y, f'{short_name}{z_note}', ha='center', va='center',
                fontsize=7, weight='bold')

    legend_patches = [mpatches.Patch(color=c, label=cat) for cat, c in colors.items()]
    ax.legend(handles=legend_patches, loc='upper right')
    ax.set_aspect('equal')
    ax.autoscale()
    margin = 500
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    ax.set_xlim(xlim[0] - margin, xlim[1] + margin)
    ax.set_ylim(ylim[0] - margin, ylim[1] + margin)
    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    ax.set_title(title or f'Scene Graph: "{obs.prompt[:50]}..."')
    ax.grid(True, alpha=0.3)

    # ── Right: screenshot (if available) ──
    ax2 = axes[1]
    if obs.rgb_views:
        view_name = list(obs.rgb_views.keys())[0]
        ax2.imshow(obs.rgb_views[view_name])
        ax2.set_title(f"Screenshot: {view_name}")
    else:
        ax2.text(0.5, 0.5, "No screenshot captured", ha='center', va='center',
                 transform=ax2.transAxes, fontsize=14, color='gray')
        ax2.set_title("Screenshot: N/A")
    ax2.axis('off')

    plt.tight_layout()
    plt.show()


def visualize_scores(results_list):
    """Bar chart comparing scores across examples."""
    names = [r[0].split(':')[0].strip() for r in results_list]
    semantic = [r[2].tier1['semantic']['score'] for r in results_list]
    physical = [r[2].tier1['physical']['score'] for r in results_list]
    structural = [r[2].tier1['structural']['score'] for r in results_list]
    composite = [r[2].composite_score for r in results_list]

    x = np.arange(len(names))
    width = 0.2

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 1.5*width, semantic, width, label='Semantic', color='#4A90D9')
    ax.bar(x - 0.5*width, physical, width, label='Physical', color='#5CB85C')
    ax.bar(x + 0.5*width, structural, width, label='Structural', color='#F0AD4E')
    ax.bar(x + 1.5*width, composite, width, label='Composite', color='#333333')

    ax.set_ylabel('Score (0-1)')
    ax.set_title('Scene Verifier Scores')
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.3, label='Good threshold')
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Poor threshold')
    plt.tight_layout()
    plt.show()


# ── Visualize all results ──
for name, obs, result in results:
    visualize_scene_topdown(obs, title=f"{name} (score: {result.composite_score:.2f})")

visualize_scores(results)

### 10. Formatted Example Output

In [ ]:
def format_example(name: str, obs: SceneObservation, result: VerifierResult,
                   explanation: str = "") -> str:
    """Format a single example in the deliverable format."""
    lines = [name]
    lines.append(f'- Input: "{obs.prompt}"')
    has_screenshot = "screenshot" if obs.rgb_views else "scene graph only"
    lines.append(f'- Output: [{has_screenshot}]')
    lines.append(f'- Score: {result.composite_score:.2f}')
    lines.append(f'- Breakdown:')
    lines.append(f'    Tier 1:  semantic={result.tier1["semantic"]["score"]:.2f}, '
                 f'physical={result.tier1["physical"]["score"]:.2f}, '
                 f'structural={result.tier1["structural"]["score"]:.2f}')
    if result.tier2:
        lines.append(f'    Tier 2:  shape={result.tier2["shape"]["score"]:.2f}, '
                     f'depth={result.tier2["depth"]["score"]:.2f}, '
                     f'perceptual={result.tier2["perceptual"]["score"]:.2f}')
    else:
        lines.append(f'    Tier 2:  N/A (no reference)')
    lines.append(f'- Diagnostics:')
    seen = set()
    for d in result.diagnostics:
        key = d[:60]
        if key not in seen:
            lines.append(f'    {d}')
            seen.add(key)
    if explanation:
        lines.append(f'- Explanation: {explanation}')
    return '\n'.join(lines)


print('=' * 70)
print('SCENE VERIFIER — EVALUATION RESULTS')
print('=' * 70)
print()

for i, (name, obs, result) in enumerate(results):
    print(format_example(f'Example {i+1}', obs, result))
    print()
    print('-' * 70)
    print()

### 11. Tier 2: Reference-Based Metrics

In [ ]:
def compute_segmentation_iou(gen_seg: np.ndarray, ref_seg: np.ndarray,
                              category_weights: dict = None) -> dict:
    """
    Compute per-category IoU between generated and reference segmentation masks.
    Activated when UE5 segmentation renders are available.
    """
    if category_weights is None:
        category_weights = {0: 0.4, 1: 0.2, 2: 0.2, 3: 0.2}

    categories = set(np.unique(gen_seg)) | set(np.unique(ref_seg))
    ious = {}
    for cat in categories:
        gen_mask = (gen_seg == cat)
        ref_mask = (ref_seg == cat)
        intersection = np.logical_and(gen_mask, ref_mask).sum()
        union = np.logical_or(gen_mask, ref_mask).sum()
        ious[int(cat)] = intersection / union if union > 0 else 0.0

    weighted_iou = sum(ious.get(c, 0) * w for c, w in category_weights.items())
    total_weight = sum(category_weights.values())

    return {
        'score': weighted_iou / total_weight if total_weight > 0 else 0.0,
        'per_category': ious,
    }


def compute_depth_score(gen_depth: np.ndarray, ref_depth: np.ndarray) -> dict:
    """Scale-invariant depth comparison + rank correlation."""
    from scipy import stats

    eps = 1e-6
    gen_log = np.log(gen_depth.clip(eps))
    ref_log = np.log(ref_depth.clip(eps))

    d = gen_log - ref_log
    n = d.size
    si_error = np.mean(d**2) - (np.sum(d)**2) / (n**2)
    si_score = max(0.0, 1.0 - si_error)

    sample_size = min(1000, n)
    indices = np.random.choice(n, sample_size, replace=False)
    gen_flat = gen_depth.flatten()[indices]
    ref_flat = ref_depth.flatten()[indices]
    rank_corr, _ = stats.spearmanr(gen_flat, ref_flat)
    rank_score = max(0.0, rank_corr)

    return {
        'score': 0.5 * si_score + 0.5 * rank_score,
        'si_error': float(si_error),
        'rank_correlation': float(rank_corr),
    }


print("Tier 2 metrics ready (activated when reference depth/segmentation data is provided).")

### 12. Reflection

In [ ]:
reflection = """
SCENE VERIFIER — REFLECTION

What works well:
- Scene graph as backbone: SimWorld provides ground-truth metadata (exact positions,
  types, bounding boxes). Structural checks (collision, floating, underground) are
  extremely reliable — no VLM hallucination risk.
- VLM-enhanced semantic parsing: Using Claude to extract requirements from prompts
  handles nuanced language ("cozy neighborhood", implicit plurals) far better than
  regex. The regex fallback ensures robustness.
- Combined physical scoring: Scene graph catches precise numerical violations
  (z=-100 = underground) while VLM catches visual issues the graph can't
  represent (weird shadows, implausible angles not captured in metadata).
- Two-tier architecture: Reference-free Tier 1 handles the primary use case
  (text → scene). Tier 2 adds precision for regression testing.

What fails:
- VLM spatial reasoning on edge cases: VLMs can struggle with precise spatial
  judgments. The scene graph compensates, but purely visual scenes without
  metadata would suffer.
- Prompt parsing coverage: Even VLM-based parsing misses deeply implied
  requirements ("cozy" implies spacing, warmth, style).
- No aesthetic evaluation: The verifier checks correctness but not beauty.
  A correctly-placed but ugly scene scores high.

What the metric captures:
- Completeness: Are all requested objects present?
- Physical validity: No floating, clipping, underground, or scale violations
- Structural integrity: Bounding box consistency, scene bounds
- Prompt fidelity: Object types, counts, and attributes match the request

What it misses:
- Aesthetic quality: Style, lighting, visual coherence
- Functional coherence: Do doors face streets? Are windows at reasonable heights?
- Temporal consistency: Does quality improve across iterative edits?
- Semantic nuance: "cozy neighborhood" vs "neighborhood"
- Agent evaluation: Tool usage efficiency, skill composition quality
"""

print(reflection)